# 第 3 章 · Agent 与工具调用：LangChain 1.x 的核心范式

> 本章目标：
> 1. 掌握 LangChain 1.x 唯一的 Agent 构建入口——**`create_agent`**；
> 2. 学会定义工具（`@tool`），并"解剖"Agent 的消息级运行过程；
> 3. 初识**中间件（Middleware）**——LangChain 世界的回调系统；
> 4. 理解 `create_agent` 与 LangGraph 的血缘关系。

---

## 1. `create_agent`：一行代码背后的图

LangChain 1.x 把过去五花八门的 Agent 执行器统一为 `create_agent`：

```python
from langchain.agents import create_agent

agent = create_agent(model=llm, tools=[...], system_prompt="...")
```

返回的 `agent` 是一个**编译好的 LangGraph 图**——这解释了为什么 LangChain 需要 LangGraph 作为依赖：

```mermaid
flowchart LR
    I(["用户输入 messages"]) --> M["🧠 model 节点<br/>调用 LLM"]
    M -->|"有 tool_calls"| T["🔧 tools 节点<br/>执行工具"]
    T -->|"ToolMessage 回填"| M
    M -->|"无 tool_calls"| E(["END 返回"])
    style M fill:#e8f0fe,stroke:#4285f4,stroke-width:2px
    style T fill:#fef7e0,stroke:#fbbc04,stroke-width:2px
```

> 📌 这就是著名的 **ReAct 循环**：模型推理（Reason）→ 调用工具（Act）→ 观察结果 → 再推理……直到模型认为无需再调工具。ADK 的 Runner 内循环与此完全同构——**所有 Agent 框架的心脏都是这颗**。

---

## 2. 定义工具：`@tool` 装饰器

LangChain 中定义工具的标准方式是 `@tool` 装饰器（1.x 也支持直接传裸函数）。与 ADK 相同：**docstring 即说明书，类型注解即参数 schema**：


In [1]:
import os
assert os.environ.get("DEEPSEEK_API_KEY"), "请先设置环境变量 DEEPSEEK_API_KEY"

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.environ["DEEPSEEK_API_KEY"],
    base_url="https://api.deepseek.com",
    temperature=0,
)

@tool
def get_stock_price(symbol: str) -> dict:
    """查询股票最新价格。

    Args:
        symbol: 股票代码，例如 "AAPL"、"TSLA"。
    """
    fake = {"AAPL": 232.5, "TSLA": 248.1}
    return {"symbol": symbol, "price": fake.get(symbol.upper(), "未收录")}

@tool
def convert_currency(amount: float, from_cur: str, to_cur: str) -> dict:
    """货币换算（演示汇率）。

    Args:
        amount: 金额。
        from_cur: 源货币，如 USD/CNY。
        to_cur: 目标货币，如 USD/CNY。
    """
    rate = {("USD", "CNY"): 7.2, ("CNY", "USD"): 1 / 7.2}
    r = rate.get((from_cur.upper(), to_cur.upper()))
    return {"result": round(amount * r, 2) if r else "不支持的货币对"}

print("工具名：", get_stock_price.name)
print("工具描述：", get_stock_price.description[:40], "...")
print("参数 schema：", list(get_stock_price.args.keys()))


工具名： get_stock_price
工具描述： 查询股票最新价格。

    Args:
        symbol: 股票代 ...
参数 schema： ['symbol']


---

## 3. 创建并运行 Agent

`create_agent(model, tools, system_prompt)` 三要素，与 ADK 的 `Agent(model, tools, instruction)` 一一对应：


In [2]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

agent = create_agent(
    model=llm,
    tools=[get_stock_price, convert_currency],
    system_prompt="你是投资小助手：需要数据时调用工具，回答简洁，金额保留两位小数。",
)

result = agent.invoke({"messages": [HumanMessage(content="我持有 10 股苹果股票，折合成人民币大概多少钱？")]})
print(result["messages"][-1].content)


我来为你计算：

**苹果股票（AAPL）**
- 最新股价：$232.50/股
- 你持有：10 股
- 总市值：$232.50 × 10 = **$2,325.00**

**换算成人民币**
- 汇率：1 USD = 7.20 CNY
- $2,325.00 × 7.20 = **¥16,740.00**

你持有的 10 股苹果股票折合人民币约为 **¥16,740.00**。


> 🎉 Agent 自己完成了"查股价 → 换算汇率"两步推理。注意输入输出都是 **`{"messages": [...]}`** 字典——这是 LangGraph 图的 State 形态（第 4 章详解）。

---

## 4. 解剖运行过程：消息列表里藏着一切

与 ADK 的事件流不同，LangChain/LangGraph 把完整过程记录在**消息列表**里。把 `result["messages"]` 全部打印出来，Agent 的"内心戏"一览无余：


In [3]:
for i, m in enumerate(result["messages"]):
    role = m.type.upper()
    if m.type == "ai" and m.tool_calls:
        for tc in m.tool_calls:
            print(f"[{i}] {role} 🔧 决定调用 {tc['name']}({tc['args']})")
    elif m.type == "tool":
        print(f"[{i}] {role} 📦 工具返回: {m.content[:60]}")
    else:
        content = m.content if isinstance(m.content, str) else str(m.content)
        print(f"[{i}] {role} 💬 {content[:80]}")


[0] HUMAN 💬 我持有 10 股苹果股票，折合成人民币大概多少钱？
[1] AI 🔧 决定调用 get_stock_price({'symbol': 'AAPL'})
[1] AI 🔧 决定调用 convert_currency({'amount': 1, 'from_cur': 'USD', 'to_cur': 'CNY'})
[2] TOOL 📦 工具返回: {"symbol": "AAPL", "price": 232.5}
[3] TOOL 📦 工具返回: {"result": 7.2}
[4] AI 💬 我来为你计算：

**苹果股票（AAPL）**
- 最新股价：$232.50/股
- 你持有：10 股
- 总市值：$232.50 × 10 = **$2,32


| 消息类型 | 在 ReAct 循环中的角色 |
|---|---|
| `HumanMessage` | 任务起点 |
| `AIMessage`（带 `tool_calls`） | 模型的"行动决策" |
| `ToolMessage` | 工具执行结果（观察） |
| `AIMessage`（纯文本） | 循环终点：最终回答 |

> 🔍 对比 ADK：ADK 的 Event 流是**推送式**的（边运行边 yield），LangGraph 的消息列表是**累积式**的（State 的一部分）。两种设计在第 4、5 章会看到更深的影响。

---

## 5. 流式运行：看见中间过程

`agent.stream()` 按图节点逐步产出状态更新——这是调试 Agent 决策过程的利器：


In [4]:
for chunk in agent.stream(
    {"messages": [HumanMessage(content="TSLA 现在多少钱一股？")]},
    stream_mode="updates",
):
    for node_name, update in chunk.items():
        msg = update["messages"][-1]
        if msg.type == "ai" and msg.tool_calls:
            print(f"⚡ 节点[{node_name}] → 调用工具 {msg.tool_calls[0]['name']}")
        elif msg.type == "tool":
            print(f"📦 节点[{node_name}] → 工具结果 {msg.content[:50]}")
        else:
            print(f"💬 节点[{node_name}] → {msg.content[:60]}")


⚡ 节点[model] → 调用工具 get_stock_price
📦 节点[tools] → 工具结果 {"symbol": "TSLA", "price": 248.1}


💬 节点[model] → 特斯拉（TSLA）最新股价为 **248.10 美元/股**。

需要我帮您换算成人民币或其他货币吗？


---

## 6. 中间件（Middleware）：LangChain 的回调系统

生产需求（护栏、审计、动态改 prompt、人工审批）通过**中间件**插入 Agent 生命周期，与 ADK 的 Callbacks 对应：

| 钩子 | 时机 | ADK 对应 |
|---|---|---|
| `before_model` | 每次调 LLM 前 | `before_model_callback` |
| `after_model` | LLM 返回后 | `after_model_callback` |
| `wrap_model_call` | 包裹 LLM 调用（可改写/缓存/重试） | 无直接对应（更灵活） |
| `wrap_tool_call` | 包裹工具调用 | `before/after_tool_callback` |

下面写一个"调用计数 + 敏感词护栏"中间件：


In [5]:
from langchain.agents.middleware import AgentMiddleware
from langchain_core.messages import AIMessage

class GuardMiddleware(AgentMiddleware):
    """before_model：统计调用次数；命中敏感词则短路回复。"""

    def __init__(self):
        self.call_count = 0
        self.blocklist = ["中奖", "赌博"]

    def before_model(self, state, runtime):
        self.call_count += 1
        last = state["messages"][-1]
        text = last.content if isinstance(last.content, str) else ""
        for word in self.blocklist:
            if word in text:
                print(f"  🛡️ [中间件] 拦截敏感词「{word}」")
                return {"messages": [AIMessage(content="抱歉，该话题我无法参与讨论。")]}
        return None

guarded_agent = create_agent(
    model=llm, tools=[], middleware=[GuardMiddleware()],
    system_prompt="你是日常助手，简洁回答。",
)
guard = guarded_agent.middleware[0] if hasattr(guarded_agent, "middleware") else None

r1 = guarded_agent.invoke({"messages": [HumanMessage(content="推荐一本关于时间管理的书")]})
print("正常问题：", r1["messages"][-1].content[:60])

r2 = guarded_agent.invoke({"messages": [HumanMessage(content="我赌博赢钱的概率怎么算？")]})
print("敏感问题：", r2["messages"][-1].content[:60])


正常问题： 《深度工作》——卡尔·纽波特。  
核心：专注深度任务，减少碎片时间，比单纯“挤时间”更有效。  
适合想提升效率、对抗
  🛡️ [中间件] 拦截敏感词「赌博」


敏感问题： 赌博是非法且有害的行为，我不能提供任何相关建议。如果你有压力或困惑，可以聊聊其他健康的话题，比如运动或学习。


> 🔍 与 ADK 回调的对照极为工整：返回 `None` 放行，返回消息则**短路**——两个框架在"护栏"这个工程刚需上给出了几乎相同的答案。

---

## 7. 先睹为快：Agent 的记忆（第 5 章详解）

`create_agent` 返回的是 LangGraph 图，因此天然支持 **Checkpointer**——挂上它，Agent 立刻拥有多轮记忆：


In [6]:
from langgraph.checkpoint.memory import InMemorySaver

chatty = create_agent(
    model=llm, tools=[],
    system_prompt="你是贴心的聊天伙伴，记性很好。",
    checkpointer=InMemorySaver(),          # ← 挂上检查点存储
)
config = {"configurable": {"thread_id": "chat-001"}}   # 线程 = 一段对话

r = chatty.invoke({"messages": [HumanMessage(content="我叫小王，是个徒步爱好者。")]}, config)
print("第1轮：", r["messages"][-1].content[:60])

r = chatty.invoke({"messages": [HumanMessage(content="根据我的爱好，推荐一个周末去处，记得带上我的名字。")]}, config)
print("第2轮：", r["messages"][-1].content)


第1轮： 嗨，小王！很高兴认识你！徒步爱好者，听起来真棒！你平时喜欢去哪里徒步呀？有没有特别推荐的路线？我虽然不能亲自去，但很喜欢


第2轮： 当然可以，小王！既然你热爱徒步，我推荐你去**北京郊外的海坨山**，那里有高山草甸和云海，特别适合周末放松。记得带上你的登山杖和相机，清晨的日出绝对值得早起！如果你想要更轻松一点的路线，**杭州的十里琅珰**也很不错，茶田和竹林交错，走起来很惬意。你觉得哪个更合你心意？


> 🔍 `thread_id` ↔ ADK 的 `session_id`，`InMemorySaver` ↔ `InMemorySessionService`——两个框架的"记忆"在此处严丝合缝地对上了。第 5 章会展开 Checkpointer 的全部玩法（时间旅行、人机协同）。

---

## 8. 与 ADK 对照 🔄

| 维度 | LangChain `create_agent` | ADK `Agent` |
|---|---|---|
| 本质 | 编译好的 **LangGraph 图** | 框架托管的**运行单元** |
| 调用方式 | `invoke({"messages": [...]})` 返回完整 State | `run_async()` 流式产出 Event |
| 工具定义 | `@tool` 装饰器 / 裸函数 | 裸函数 / `FunctionTool` |
| 过程可见性 | 消息列表累积在 State 中 | Event 流逐条推送 |
| 护栏机制 | Middleware（可组合链式） | Callbacks（六种钩子） |
| 记忆 | Checkpointer（可选挂载） | SessionService（内建必需） |

---

## 📌 本章要点回顾

- LangChain 1.x 的 Agent = `create_agent` 产出的 **ReAct 循环图**；
- 运行过程完整记录在消息列表：`HumanMessage → AIMessage(tool_calls) → ToolMessage → AIMessage`；
- `stream(stream_mode="updates")` 是观察节点级过程的最佳姿势；
- **中间件**实现护栏/审计，语义与 ADK 回调一致（None 放行、非 None 短路）；
- 挂 `checkpointer` + `thread_id` 即刻获得多轮记忆。

> ➡️ 下一章：[04-LangGraph核心-图与状态](04-LangGraph核心-图与状态.ipynb) —— 揭开 `create_agent` 的引擎盖，直接驾驶图引擎。


---

## 🧪 本章练习

### 1. 构建并解剖 ReAct Agent（基础）

定义“商品查询”和“运费计算”两个工具，使用 `create_agent` 回答组合问题。工具必须有清晰 schema、参数校验与统一错误返回。打印最终消息列表，标注用户消息、模型的 `tool_calls`、工具结果和最终回答，说明一次 ReAct 循环是如何闭合的。

### 2. 流式过程查看器（进阶）

基于 `stream(stream_mode="updates")` 编写命令行查看器，以易读格式展示当前节点、工具名、参数、结果摘要和最终回答；敏感参数必须脱敏。验收时让 Agent 连续调用两个工具，并确保用户能区分“过程更新”和“最终答案”。

### 3. 用 Middleware 建立护栏与审计（工程）

实现至少两个可组合中间件：一个限制危险请求或工具参数，一个记录模型/工具耗时与错误。准备允许、拒绝、工具异常三类用例，验证中间件顺序不会导致被拒绝的操作仍然执行，并对照 ADK callback 的 `None` 放行/非 `None` 短路语义。

### 4. 多线程记忆与隔离（进阶）

给 Agent 挂载 Checkpointer，创建两个 `thread_id` 并分别保存不同偏好。连续调用后自动断言同一线程能够回忆、不同线程不能串话；再故意复用错误的 `thread_id`，记录问题并提出服务端生成与校验 thread ID 的方案。

### 5. 工具可靠性设计（开放）

假设工具会超时、返回脏数据或产生不可逆副作用，为本章 Agent 设计最大步骤数、超时、重试、熔断、幂等键和人工确认策略。说明哪些失败可由模型自我修复，哪些必须由确定性代码接管；选择其中两项实现并用故障注入验证。
